In [1]:
import time
from datetime import datetime
import os, re
import pandas as pd
from pathlib import Path
from utilities import timediff, osprey, xlsToXlsx
# %run utilities.ipynb

In [2]:
# get input data
start_time          = time.time()
start_time_overlord = time.time()
print('Determining latest cash recon files ...')
cash_folder  = r'P:\Investment Operations\GRC\Compliance\Overdrafts' # where the final files will be stored
aax_folder   = r'\\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\Daily\fund_codes.xlsx'
py_folder    = r'P:\Investment Operations\GRC\Compliance\Daily\py_reports.xlsm'
pth_download = str(Path.home() / "Downloads")

# get the names of the just saved PIM and PFSI cash recon files
pim_f        = max([s[:8] for s in os.listdir(cash_folder) if 'PIM'  in s]) + '_PIM_Cash_Recon.xlsx'
pfsi_f       = max([s[:8] for s in os.listdir(cash_folder) if 'PFSI' in s]) + '_PFSI_Cash_Report.xls'

# get list of bank codes across Portal, Eage bank summary, and cash recon
banks        = pd.read_excel(aax_folder, sheet_name = 'Sttlmnt', usecols = 'G:J').dropna()
pfsi_codes   = pd.read_excel(py_folder , sheet_name = 'funds',   usecols = 'V:W').dropna()

print('\n', f' Latest overdraft file names:','\n', f'  {pim_f} for PIM, and', '\n', f'  {pfsi_f} for PFSI.', '\n') # print the latest file names
print(f'Determining latest cash recon files completed: {timediff(start_time, time.time())}', '\n')

Determining latest cash recon files ...

  Latest overdraft file names: 
   20250824_PIM_Cash_Recon.xlsx for PIM, and 
   20250824_PFSI_Cash_Report.xls for PFSI. 

Determining latest cash recon files completed: 4.0sec 



In [3]:
# get the pfsi and pim fund codes to look up
start_time = time.time()
print('Creating dataframes and determining the PIM and PFSI funds for which NAVS must be looked up ...', '\n')

# getting report parameters
import xlwings as xw
pth_py_reports = r'P:\Investment Operations\GRC\Compliance\Daily\py_reports.xlsm'
wb             = xw.Book(pth_py_reports) # open calc workbook as an object
aladdin        = wb.sheets['creds'].range('A1').value
sesame         = wb.sheets['creds'].range('A2').value
wb.close()

# convert the pfsi file format from .xls to .xlsx
xlsToXlsx(os.path.join(cash_folder, pfsi_f)) # calling function from utilities.py

# dataframing the overdraft sheets
# https://stackoverflow.com/questions/17977540/pandas-looking-up-the-list-of-sheets-in-an-excel-file
xl   = pd.ExcelFile( os.path.join(cash_folder, pim_f))
pim  = pd.read_excel(os.path.join(cash_folder, pim_f), sheet_name = xl.sheet_names[0], usecols = 'A:I,O', header = 11)
pfsi = pd.read_excel(os.path.join(cash_folder, pfsi_f) + 'x')

# find the row with the bank balance heading  https://stackoverflow.com/questions/26640129/
# search-for-string-in-all-pandas-dataframe-columns-and-filter
# k = pim[pim.eq('Closing Balance BNK').any(axis = 1)].index[0]

# get the pim fund codes to look up
pim_fnds_  = ','.join((pim['Client ID'].unique()))

# get the pfsi funds codes to look up
pfsi_codes = pd.read_excel(pth_py_reports,sheet_name = 'funds', usecols = 'W:Z').dropna()
pfsi_fnds_ = (',').join(pd.merge(pfsi, pfsi_codes, left_on = 'Account Number', right_on = 'NTRS Codes', how = 'left')['PIM Codes'].unique())

# get the overdraft reporting dates
xl        = pd.ExcelFile(cash_folder + str('\\') + pim_f)
pim_date  = datetime.strptime(pim.iloc[0,0], "%d/%m/%Y")
pfsi_date = pfsi['D-VALN-AS-OF'].iloc[0]

print(f' Dataframes have shapes PIM: {pim.shape} and PFSI: {pfsi.shape}','\n')
print(f' PIM:  {pim_date.strftime("%a %d %B %Y")} for {len(pim_fnds_.split(","))} funds -' , '\n', f'{pim_fnds_}' , '\n\n',
      f'PFSI: {pfsi_date.strftime("%a %d %B %Y")} for {len(pfsi_fnds_.split(","))} funds -', '\n', f'{pfsi_fnds_}', '\n')
print(f'{timediff(start_time, time.time())} creating dataframe and determining the PIM and PFSI funds for which NAVs \
must be looked up  \n')

Creating dataframes and determining the PIM and PFSI funds for which NAVS must be looked up ... 

  P:\Investment Operations\GRC\Compliance\Overdrafts\20250824_PFSI_Cash_Report.xlsx
 Dataframes have shapes PIM: (208, 10) and PFSI: (48, 12) 

 PIM:  Fri 22 August 2025 for 162 funds - 
 3BBCIINC,ABMMBND,ADRRC,ADVMB,ADVMM,AFPFLB,AMMARF,AMPBQP,ASBTOS,ASHFLX,BCIFIF,BPROV,CCNPF,CCTCSH,CMPFCASH,CMPFFLEX,CMPFINC,CSIRBQP,ECICBAL,ELCIPF,ENGENIP,ENGENMBF,FEMEF,FEMPBF,GACASH,GAEMBF,GEMSMED,GMRETF,GMRETF2,GRFINV,GTCWP2,HOLADC,HOLDINC,HOLIDC,HOLMMF,HOLYPF,HOSMED,IJGBAL,IJGCOR,IJGIPF,IJGMMF,IMPALA,IMPBAL,IMPREF,IPCASH,IPIPF,ISPFP,LEZAFFI,LEZALDI,LIBTAA,LNBINS,LNBLIN,LPIIFTAA,MASAINC,MASAMMF,MASASI,MEDINC,MOMABIL,MOMBBF,MOMFLX,MOMFXB,MOMIPF,MOMPRET,MOMTAAHI,MOMTAALI,MOMTAAMI,MULTICH,MWPFEQU,MWPFILB,MYQIP,NEDMED,NELSON,NESEQU,NFMWAGG,NFMWEQU,NFMWGF,NFMWSTA,NGKINC,OMLMMF,OMMAIF,PABS,PBNDQ,PCAEF,PCBF,PCCEF,PCEQTF,PCGEF,PCMMF,PCSHQ,PEQ,PEQF,PETFIP,PEYF,PFFIF,PGARFF,PGBF,PGIF,PGIPFA,PICPROV,PIF,PIMBAL,PIME

In [4]:
# get the PIM fund NAVs as at the PIM overdrafts report date

    # get the PIM NAV.csv file ...
rpt            = 'fnav'
xtnsn          = 'csv'
osprey(rpt, pim_fnds_, pim_date, pim_date, '', xtnsn, aladdin, sesame)

    # ... and then immediately dataframe it
pim_navs_name  = f'{rpt.upper()} ({len(pim_fnds_.split(","))}) {pim_date.strftime("%d%b%Y")}.{xtnsn}'
pim_navs       = pd.read_csv(os.path.join(pth_download, pim_navs_name),
                             usecols = ['Effective Date', 'NAV Entity ID', 'Total Net Assets'])
# https://stackoverflow.com/questions/10591000/specifying-data-type-in-pandas-csv-reader
# https://www.reddit.com/r/learnpython/comments/5ktuhv/how_do_i_remove_commas_from_data_frame_column/
pim_navs['Total Net Assets'] = pim_navs['Total Net Assets'].str.replace(',','').astype(float)
pim_navs.rename(columns = {'Total Net Assets': 'NAV'}, inplace = True)

In [5]:
# compile and save the PIM cash % of NAV dataframe

start_time = time.time()
print('Compiling and saving the PIM funds cash % of NAV sheet ...')
# https://stackoverflow.com/questions/71041451/pandas-divide-values-in-one-dataframe-by-corresponding-values-from-another-data
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html
pim_od  = pim.merge(pim_navs, how = 'inner', left_on  = 'Client ID', right_on = 'NAV Entity ID')
pim_od['Closing Balance BNK % of NAV'] = pim_od['Closing Balance BNK'] / pim_od['NAV'] * 100

# reorder the headings
headers = ['Value Date', 'Report Date', 'Client ID', 'Closing Balance BNK', 'Closing Balance BNK % of NAV',
           'Closing Balance MGR', 'NAV', 'Currency', 'Custodian']
pim_od  = pim_od[headers]

# sort by the % of NAV col
pim_od  = pim_od.sort_values(by = ['Closing Balance BNK % of NAV'], ascending=True)

# write the pim df to the overdrafts folder
pim_od.to_excel(os.path.join(cash_folder, pim_date.strftime("%Y%m%d") + '_PIM_Fund_Bank_Recons.xlsx'),
      sheet_name = f'PIM ' + pim_date.strftime("%d%b%Y"), index = False)

start_time = time.time()
print(f'{timediff(start_time, time.time())} compiling and saving the PIM funds cash % of NAV sheet \n')

Compiling and saving the PIM funds cash % of NAV sheet ...
0.0sec compiling and saving the PIM funds cash % of NAV sheet 



In [6]:
# Prettify the PIM sheet

print(f'Prettifying and adding links to the PIM cash sheet and then saving it ...')
start_time = time.time()

wbS    = xw.Book(os.path.join(cash_folder, pim_date.strftime("%Y%m%d")) + '_PIM_Fund_Bank_Recons.xlsx')
shtS   = wbS.sheets['PIM ' + pim_date.strftime("%d%b%Y")]  # PIM cash vs NAVs sheet

# add a filter
shtS.used_range.api.AutoFilter(Field := 1)

# add link to the cash recon folder
shtS['J1'].add_hyperlink(cash_folder, 'Cash recons')

# add conditional formating for values that are negative or exceed 100% of NAV
for cell in shtS['E2'].expand('down'):
    if cell.value < 0:
        cell.font.color = (255,   0,   0) # red is (255,0,0) in RGB or #FF0000 in Hex  
    if cell.value > 25:
        cell.font.bold  = True
        cell.color      = "#FFCC00"

# numbers in thosuands and decimals format
shtS['D:G'].number_format = "#,##0.00" # https://stackoverflow.com/questions/55391542/adjust-number-formatting-in-excel-via-xlwings-from-python
shtS['E:E'].number_format = "#,##0.000"

# set column widths
widths = {'A': 9.83, 'B': 9.83, 'C': 9.83, 'D': 15.18, 'E': 7.73, 'F': 15.18, 'G': 15.18, 'H': 4.82, 'I': 10.81, 'J': 10.81}
for col in widths:
    shtS[f'{col}' + '1'].column_width = widths[col]
    
# format the headings row of the Summary file
shtS['A1:J1'].api.WrapText = True
shtS['A1:J1'].font.bold    = True
shtS['A1:J1'].color        = (242, 242, 242) # light grey for the column headings

# https://stackoverflow.com/questions/63077985/xlwings-set-cell-formatting-from-python-on-a-mac-specifically
# left align: -4131, centre align: -4108, right align: -4152
shtS['D:G'].api.HorizontalAlignment = -4152 # right align the numerical values
shtS['1:1'].api.HorizontalAlignment = -4108 # centre align the headings

# freeze panes at cell 'B2'
wndW = wbS.app.api.ActiveWindow
wndW.FreezePanes = False
wndW.SplitColumn = 2
wndW.SplitRow    = 1
wndW.FreezePanes = True

# add a filter
shtS.used_range.api.AutoFilter(Field := 1)

wbS.save()
wbS.close()
print(f'{timediff(start_time, time.time())} prettifying and adding links to the summary sheet and then saving it \n')

Prettifying and adding links to the PIM cash sheet and then saving it ...
3.9sec prettifying and adding links to the summary sheet and then saving it 



In [7]:
# get the PFSI fund NAVs at the PFSI overdrafts report date

    # get the PFSI NAV .csv file ...
rpt            = 'fnav'
xtnsn          = 'csv'
osprey(rpt, pfsi_fnds_, pfsi_date, pfsi_date, '', xtnsn, aladdin, sesame)

    # ... and then immediately dataframe it
pfsi_navs_name = f'{rpt.upper()} ({len(pfsi_fnds_.split(","))}) {pfsi_date.strftime("%d%b%Y")}.{xtnsn}'
pfsi_navs      = pd.read_csv(os.path.join(pth_download, pfsi_navs_name))
pfsi_navs['Total Net Assets'] = pfsi_navs['Total Net Assets'].str.replace(',','').astype(float)

In [8]:
# compile and save the PFSI cash % of NAV dataframe
start_time = time.time()

print('Compiling and saving the PFSI funds cash % of NAV sheet ...')
# https://stackoverflow.com/questions/71041451/pandas-divide-values-in-one-dataframe-by-corresponding-values-from-another-data
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html
# merge with fund codes
pfsi_c  = pfsi.merge(pfsi_codes, how = 'inner', left_on  = 'Account Number', right_on = 'NTRS Codes')

# merge PFSI cash balances withfund NAVs
pfsi_od = pfsi_c.merge(pfsi_navs, how = 'outer', left_on = 'PIM Codes', right_on = 'NAV Entity ID') #how = 'inner',

# caluclate cash as % of NAV
pfsi_od['Total Cash % of NAV'] = pfsi_od['Total Cash'] / pfsi_od['Total Net Assets'] * 100

# reorder the headings
pfsi_od.rename(columns = {'Total Net Assets': 'NAV', 'NAV Entity ID': 'Fund', 'PIM Names': 'Fund Name', 'Currency code': 'Crncy'}, inplace = True)
#headers = ['Consolidation', 'Account Number', 'Account name', 'D-VALN-AS-OF', 'Currency code', 'Currency name - asset', 'C-ROW-TYPE-CODE', 
           #'Liquid Cash', 'Invested Cash', 'Total Cash', 'Total Cash % of NAV', 'NAV', 'Fund', 'Fund Name']
headers = ['Account Number', 'D-VALN-AS-OF', 'Crncy', 'Liquid Cash', 'Invested Cash', 
           'Total Cash', 'NAV', 'Total Cash % of NAV', 'Fund', 'Fund Name']
pfsi_od = pfsi_od[headers]

# convert the numerical columns from type 'object' to type 'float' ...
#cols_format = ['Liquid Cash', 'Invested Cash', 'Total Cash', 'Total Cash % of NAV', 'NAV']

# sort by the % of NAV col
pfsi_od = pfsi_od.sort_values(by = ['Fund', 'Total Cash % of NAV'], ascending = [True, True])

# write the pfsi df to the overdrafts folder
pfsi_od.to_excel(os.path.join(cash_folder, pfsi_date.strftime("%Y%m%d") + '_PFSI_Fund_Bank_Recons.xlsx'),
      sheet_name = f'PFSI ' + pfsi_date.strftime("%d%b%Y"), index = False)

start_time = time.time()
print('\n', len(pfsi), len(pfsi_od), '\n')
print(f'{timediff(start_time, time.time())} compiling and saving the PFSI funds cash % of NAV sheet \n')

Compiling and saving the PFSI funds cash % of NAV sheet ...

 48 48 

0.0sec compiling and saving the PFSI funds cash % of NAV sheet 



In [9]:
# Prettify the PFSI sheet

print(f'Prettifying and adding links to the summary sheet and then saving it ...')
start_time = time.time()

wbS    = xw.Book(os.path.join(cash_folder, pfsi_date.strftime("%Y%m%d")) + '_PFSI_Fund_Bank_Recons.xlsx')
shtS   = wbS.sheets[f'PFSI ' + pfsi_date.strftime("%d%b%Y")]  # cash vs NAVs sheet

# add a filter
shtS.used_range.api.AutoFilter(Field := 1)

# add link to the cash recon folder
shtS['J1'].add_hyperlink(cash_folder, 'Cash recons')

# make a dataframe to enable summing by fund account number
pfsi_df = pd.read_excel(os.path.join(cash_folder, pfsi_date.strftime("%Y%m%d")) + '_PFSI_Fund_Bank_Recons.xlsx')

# sum cash for a single fund
# https://docs.xlwings.org/en/stable/api/range.html
# https://stackoverflow.com/questions/74872210/how-to-fill-the-empty-cell-with-value-above-cell-using-xlwings
for cell in shtS['J2'].expand('down'):
    if cell.offset(0, -9).value != cell.offset(-1, -9).value: # if cell left doesn't equal left above ... syntax: cell.offset(row, column))
        cell.offset(0, 1).value = pfsi_df[pfsi_df['Account Number'] == cell.offset(0, -9).value]['Total Cash % of NAV'].sum()

# add conditional formating for values that are negative or exceed 20% of NAV
for cell in shtS['J2'].expand('down'):
    if cell.offset(0, 1).value is not None:
        if cell.offset(0, 1).value < 0:
            cell.offset(0, 1).font.color = (255,   0,   0) # red is (255,0,0) in RGB or #FF0000 in Hex  
        if cell.offset(0, 1).value > 20:
            cell.offset(0, 1).font.bold  = True
            cell.offset(0, 1).color      = "#FFCC00"

# numbers in thosuands and decimals format
shtS['B:B'].number_format = "yyyy-mm-dd"
shtS['D:G'].number_format = "#,##0.00" # https://stackoverflow.com/questions/55391542/adjust-number-formatting-in-excel-via-xlwings-from-python
shtS['H:H'].number_format = "#,##0.000"
shtS['K:K'].number_format = "#,##0.000"

shtS['K1'].value = "% NAV"

# set column widths
widths = {'A': 8.14, 'B': 10.73, 'C': 4.86, 'D': 16.43, 'E': 16.43, 'F': 16.43, 'G': 17.86, 'H': 10, 'I': 8.57, 'J': 52, 'K': 8.43}
for col in widths:
    shtS[f'{col}' + '1'].column_width = widths[col]
    
# format the headings row of the Summary file
shtS['A1:J1'].api.WrapText = True
shtS['A1:J1'].font.bold    = True
shtS['A1:J1'].color        = (242, 242, 242) # light grey for the column headings

# https://stackoverflow.com/questions/63077985/xlwings-set-cell-formatting-from-python-on-a-mac-specifically
# left align: -4131, centre align: -4108, right align: -4152
shtS['D:H'].api.HorizontalAlignment = -4152 # right align the numerical values
shtS['1:1'].api.HorizontalAlignment = -4108 # centre align the headings
#shtS['K:K'].api.HorizontalAlignment = -4131 # left align the cash folder link

# freeze panes at cell 'B2'
wndW             = wbS.app.api.ActiveWindow
wndW.FreezePanes = False
wndW.SplitColumn = 2
wndW.SplitRow    = 1
wndW.FreezePanes = True

wbS.save()
wbS.close()
print(f'{timediff(start_time, time.time())} prettifying and adding links to the summary sheet and then saving it\n')

print(f' Roundtrip time for the', '\n', f'  PIM  ({pim_date.strftime("%a %d %b %Y")}), and ', '\n', 
      f'  PFSI ({pfsi_date.strftime("%a %d %b %Y")})', '\n', f'cash vs NAVs sheets: {timediff(start_time_overlord, time.time())}', '\n')

# P:\Investment Operations\GRC\Compliance\Overdrafts\

Prettifying and adding links to the summary sheet and then saving it ...
8.6sec prettifying and adding links to the summary sheet and then saving it

 Roundtrip time for the 
   PIM  (Fri 22 Aug 2025), and  
   PFSI (Fri 22 Aug 2025) 
 cash vs NAVs sheets: 1min 59.8sec 



In [10]:
print(f' Roundtrip time for the', '\n', f'  PIM  ({pim_date.strftime("%a %d %b %Y")}), and ', '\n', 
      f'  PFSI ({timediff(start_time_overlord, time.time())} {pfsi_date.strftime("%a %d %b %Y")})', '\n', f'cash vs NAVs \n')

# P:\Investment Operations\GRC\Compliance\Overdrafts\

 Roundtrip time for the 
   PIM  (Fri 22 Aug 2025), and  
   PFSI (1min 59.8sec Fri 22 Aug 2025) 
 cash vs NAVs 

